# 02.01 — Enregistrer une session de télémétrie

Enregistre des mesures du simulateur dans `Code/datasets/session_*/telemetry.jsonl`. **Ne conduit jamais la voiture** et ne produit aucune imitation. Ne pas ouvrir un autre client UDP en parallèle.

In [6]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github 
Code : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code 
Python : c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [7]:
import socket, json, ast, time
HOST='127.0.0.1'; PORT_UDP=2345; PORT_TCP=2347

def management(command,timeout=4):
    if command not in ('get_config','get_static_info'):
        raise ValueError('Cette fonction attend une commande qui retourne des donnees.')
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall(command.encode('ascii'))
        data=b''
        deadline=time.monotonic()+timeout
        while time.monotonic()<deadline and len(data)<1024*1024:
            try: chunk=s.recv(65536)
            except socket.timeout: break
            if not chunk: break
            data+=chunk
            try: return json.loads(data)
            except (ValueError,UnicodeError): pass
            try: return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeError): pass
    raise RuntimeError('Reponse management absente/incomplete pour '+command+': '+repr(data[:150]))

def request_reset():
    # Le protocole sensors_par NE renvoie AUCUNE reponse a la commande reset.
    with socket.create_connection((HOST,PORT_TCP),timeout=4) as s:
        s.settimeout(4)
        s.sendall(b'reset')
        # Fermer la connexion : le serveur traite la commande sans ACK.

def open_telemetry(timeout=6):
    sock=socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    sock.settimeout(.30)
    sock.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    try:
        while time.monotonic()<end:
            sock.send(b'connect')
            try: data=sock.recv(2**18)
            except socket.timeout: continue
            if data==b'identified': return sock
        raise TimeoutError('Le plugin UDP 2345 ne repond pas : lancer une session de conduite.')
    except Exception:
        sock.close()
        raise

def next_packet(sock,timeout=3,after_steps=None):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        try: data=sock.recv(2**18)
        except socket.timeout: continue
        if data==b'disconnect': raise ConnectionError('Connexion telemetrie interrompue')
        if data==b'identified': continue
        try: packet=json.loads(data)
        except (ValueError,UnicodeError): continue
        if not isinstance(packet,dict): continue
        if after_steps is not None and int(packet.get('steps',-1))<=int(after_steps): continue
        packet['_received_unix']=time.time()
        return packet
    raise TimeoutError('Aucun nouvel echantillon telemetrie dans le delai imparti.')

def close_telemetry(sock):
    if sock is None: return
    try: sock.send(b'disconnect')
    except OSError: pass
    sock.close()

In [8]:
from datetime import datetime
from pathlib import Path
import json,time
DUREE_SECONDES=30
MAX_PAQUETS=3000
SESSION=PROJECT/'datasets'/('session_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
assert 0<DUREE_SECONDES<=600
assert 0<MAX_PAQUETS<=50000
print('Dossier cible :',SESSION)
config=management('get_config')
static=management('get_static_info')
print('Session :',static.get('TrackName'),static.get('TrackConfiguration'),static.get('CarName'))
print('Frequence ego :',config.get('ego_sampling_freq'),'Hz')

Dossier cible : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\datasets\session_20261116_201248
Session : ks_barcelona layout_gp abarth500
Frequence ego : 25 Hz


In [9]:
LANCER_ACQUISITION=True  # passer explicitement a True, jeu lance en piste
if not LANCER_ACQUISITION:
    print('Acquisition DESACTIVEE. Mettre LANCER_ACQUISITION=True pour enregistrer.')
else:
    SESSION.mkdir(parents=True,exist_ok=False)
    (SESSION/'session.json').write_text(json.dumps({'type':'telemetry_only',
        'created_at':datetime.now().isoformat(),'duration_s':DUREE_SECONDES,
        'config':config,'static':static},indent=2,ensure_ascii=False,default=str),encoding='utf8')
    s=None;count=0;t0=time.monotonic()
    try:
        s=open_telemetry()
        with (SESSION/'telemetry.jsonl').open('w',encoding='utf8') as f:
            while count<MAX_PAQUETS and time.monotonic()-t0<DUREE_SECONDES:
                raw=next_packet(s,timeout=3)
                f.write(json.dumps(raw,ensure_ascii=False,default=str)+'\n')
                count+=1
    finally:
        close_telemetry(s)
    print('Paquets:',count,'\nEnregistre dans:',SESSION)

Paquets: 1012 
Enregistre dans: C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\datasets\session_20261116_201248
